# Bài 53: Hỏi đáp thị giác (VQA)

CLIP ở Bài 52 so khớp một tập lời nhắc văn bản *cố định* với một ảnh. **Hỏi đáp thị giác (Visual Question Answering — VQA)** đòi hỏi thứ gì đó linh hoạt hơn: cho một ảnh và một câu hỏi bằng ngôn ngữ tự nhiên tùy ý về nó, hãy đưa ra một câu trả lời. Đây là cơ chế đằng sau các trợ lý đa phương thức hiện đại có thể nhìn một bức ảnh và trả lời câu hỏi về nó &mdash; một năng lực của VLM sinh (generative VLM), khác với việc so khớp tương phản của CLIP. Bài này xây dựng phép đơn giản hóa kinh điển (và vẫn được dùng rộng rãi): coi VQA là phân loại trên một tập từ vựng nhỏ, đóng gồm các câu trả lời có thể, hợp nhất một bộ mã hóa ảnh và một bộ mã hóa câu hỏi trước dự đoán cuối cùng.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Các cảnh với cặp (câu hỏi, câu trả lời)

Các cảnh chứa 0-3 hình dấu cộng/hình tròn. Mỗi cảnh nhận một trong bốn mẫu câu hỏi: "hình nào có trong ảnh" (chỉ có nghĩa khi có đúng một hình), "có bao nhiêu hình", "có hình tròn trong ảnh không", "có dấu cộng trong ảnh không" — với câu trả lời đúng được tính trực tiếp từ dữ liệu chuẩn của cảnh. Không gian câu trả lời là một tập từ vựng nhỏ, cố định: tên hình, số đếm và có/không.

In [ ]:
SIZE = 32

QUESTION_VOCAB = ['<pad>', 'what', 'shape', 'is', 'this', 'how', 'many', 'shapes', 'are', 'there',
                   'a', 'circle', 'plus', 'in', 'the', 'image']
Q2ID = {w: i for i, w in enumerate(QUESTION_VOCAB)}
ANSWER_VOCAB = ['plus', 'circle', '0', '1', '2', '3', 'yes', 'no']
A2ID = {w: i for i, w in enumerate(ANSWER_VOCAB)}

def make_shape(img, shape_type, cx, cy, r=4):
    yy, xx = np.mgrid[0:img.shape[0], 0:img.shape[1]]
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-r:cx+r+1] = 1.0
        img[cy-r:cy+r+1, cx-1:cx+2] = 1.0
    else:
        img[((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2] = 1.0
    return img

def make_scene(rng, size=SIZE, max_shapes=3):
    n = rng.integers(0, max_shapes + 1)
    img = np.zeros((size, size), dtype=np.float32)
    shapes = []
    tries = 0
    while len(shapes) < n and tries < 30:
        tries += 1
        cx, cy = rng.integers(6, size - 6), rng.integers(6, size - 6)
        if any(abs(cx - sx) < 9 and abs(cy - sy) < 9 for sx, sy, _ in shapes):
            continue
        st = rng.choice(['plus', 'circle'])
        make_shape(img, st, cx, cy)
        shapes.append((cx, cy, st))
    img = np.clip(img + rng.normal(0, 0.03, img.shape), 0, 1).astype(np.float32)
    return img, shapes

def make_qa(rng, shapes):
    n_circle = sum(1 for _, _, t in shapes if t == 'circle')
    n_plus = sum(1 for _, _, t in shapes if t == 'plus')
    qtype = rng.choice(['what_shape', 'count', 'is_there_circle', 'is_there_plus'])
    if qtype == 'what_shape':
        words = ['what', 'shape', 'is', 'in', 'the', 'image']
        answer = shapes[0][2] if len(shapes) == 1 else str(min(len(shapes), 3))
    elif qtype == 'count':
        words = ['how', 'many', 'shapes', 'are', 'there', 'in', 'the', 'image']
        answer = str(len(shapes))
    elif qtype == 'is_there_circle':
        words = ['is', 'there', 'a', 'circle', 'in', 'the', 'image']
        answer = 'yes' if n_circle > 0 else 'no'
    else:
        words = ['is', 'there', 'a', 'plus', 'in', 'the', 'image']
        answer = 'yes' if n_plus > 0 else 'no'
    return words, [Q2ID[w] for w in words], A2ID[answer]

def pad_question(ids, max_len=8):
    return ids + [0] * (max_len - len(ids))  # 0 = <pad>, distinct from every real word

rng = np.random.default_rng(4)
N = 1000
imgs, questions, answers, question_words = [], [], [], []
for _ in range(N):
    img, shapes = make_scene(rng)
    words, q_ids, a_id = make_qa(rng, shapes)
    imgs.append(img)
    questions.append(pad_question(q_ids))
    answers.append(a_id)
    question_words.append(words)
imgs = np.array(imgs, dtype=np.float32)
questions = np.array(questions, dtype=np.int64)
answers = np.array(answers, dtype=np.int64)

split = int(0.85 * N)
Xtr, Qtr, Atr = imgs[:split], questions[:split], answers[:split]
Xte, Qte, Ate = imgs[split:], questions[split:], answers[split:]

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, im, words, a in zip(axes, imgs[:4], question_words[:4], answers[:4]):
    ax.imshow(im, cmap='gray'); ax.axis('off')
    ax.set_title(f'{" ".join(words)}?\n-> {ANSWER_VOCAB[a]}', fontsize=7)
plt.show()

## Hợp nhất một bộ mã hóa ảnh và một bộ mã hóa câu hỏi

Cả hai bộ mã hóa đều ánh xạ tới vector cùng kích thước, theo cùng mẫu như CLIP ở Bài 52. Khác biệt là điều xảy ra tiếp theo: CLIP so sánh vector nhúng ảnh và văn bản bằng tích vô hướng để kiểm tra *chúng có khớp nhau không*. VQA cần thực sự *kết hợp* chúng thành một biểu diễn chung và phân loại — một cách hợp nhất (fusion) tiêu chuẩn, đơn giản là phép nhân từng phần tử (vector ảnh và vector câu hỏi điều tiết lẫn nhau), theo sau là một phần đầu phân loại nhỏ trên tập từ vựng câu trả lời cố định.

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1))
        self.proj = nn.Linear(32, out_dim)

    def forward(self, x):
        return self.proj(self.conv(x).flatten(1))

class QuestionEncoder(nn.Module):
    def __init__(self, vocab_size, out_dim=32, embed_dim=16):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.proj = nn.Linear(embed_dim, out_dim)

    def forward(self, ids):
        mask = (ids != 0).float().unsqueeze(-1)  # ignore <pad> tokens when pooling
        emb = self.embed(ids) * mask
        pooled = emb.sum(1) / mask.sum(1).clamp(min=1)
        return self.proj(pooled)

class VQAModel(nn.Module):
    def __init__(self, vocab_size, n_answers, dim=32):
        super().__init__()
        self.img_enc = ImageEncoder(dim)
        self.q_enc = QuestionEncoder(vocab_size, dim)
        self.head = nn.Sequential(nn.Linear(dim, 32), nn.ReLU(), nn.Linear(32, n_answers))

    def forward(self, img, q_ids):
        fused = self.img_enc(img) * self.q_enc(q_ids)
        return self.head(fused)

torch.manual_seed(0)
model = VQAModel(len(QUESTION_VOCAB), len(ANSWER_VOCAB))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); Qt = torch.tensor(Qtr); At = torch.tensor(Atr)
for _ in range(300):
    opt.zero_grad()
    loss = F.cross_entropy(model(Xt, Qt), At)
    loss.backward()
    opt.step()

with torch.no_grad():
    preds = model(torch.tensor(Xte).unsqueeze(1), torch.tensor(Qte)).argmax(1).numpy()
acc = (preds == Ate).mean()

most_common = np.bincount(Atr).argmax()
baseline_acc = (Ate == most_common).mean()

print(f'VQA test accuracy: {acc:.1%}')
print(f'baseline (always predict the most common training answer, "{ANSWER_VOCAB[most_common]}"): {baseline_acc:.1%}')

## Phép thử thực sự: cùng một ảnh có nhận được các câu trả lời khác nhau cho các câu hỏi khác nhau không?

Một mô hình âm thầm bỏ qua câu hỏi và chỉ so khớp mẫu trên ảnh sẽ đưa ra cùng một câu trả lời bất kể được hỏi gì. Hãy đặt cả bốn loại câu hỏi cho một ảnh kiểm tra cố định và kiểm tra.

In [ ]:
test_rng = np.random.default_rng(123)
demo_img, demo_shapes = make_scene(test_rng)
while len(demo_shapes) != 2 or demo_shapes[0][2] == demo_shapes[1][2]:
    demo_img, demo_shapes = make_scene(test_rng)  # find a scene with one of each shape

demo_questions = [
    ['how', 'many', 'shapes', 'are', 'there', 'in', 'the', 'image'],
    ['is', 'there', 'a', 'circle', 'in', 'the', 'image'],
    ['is', 'there', 'a', 'plus', 'in', 'the', 'image'],
]
img_t = torch.tensor(demo_img[None, None])
print(f'true scene contents: {[s[2] for s in demo_shapes]}')
with torch.no_grad():
    for words in demo_questions:
        q_ids = torch.tensor([pad_question([Q2ID[w] for w in words])])
        pred = model(img_t, q_ids).argmax(1).item()
        print(f'  "{" ".join(words)}?" -> {ANSWER_VOCAB[pred]}')

Cùng một ảnh cố định tạo ra ba câu trả lời khác nhau, mỗi câu đều đúng, tùy thuộc vào câu hỏi được đặt ra — bước hợp nhất thực sự đang đọc cả hai đầu vào, không chỉ ghi nhớ các lối tắt từ ảnh đến câu trả lời.

## Từ phân loại từ vựng đóng đến VQA sinh thực sự

Coi VQA là phân loại trên một tập câu trả lời nhỏ cố định (như bài này đã làm) là cách tiếp cận thống trị trong nhiều năm và vẫn được dùng khi không gian câu trả lời thực sự nhỏ và đã biết trước. Nó thất bại ngay khi một câu trả lời cần là văn bản mở — "hãy mô tả điều gì bất thường trong ảnh này", hoặc một từ trả lời chưa bao giờ xuất hiện trong huấn luyện. Các hệ thống VQA và tạo chú thích ảnh hiện đại giải quyết điều này theo cách bộ giải mã ở Bài 46 làm: thay phần đầu phân loại bằng một bộ giải mã văn bản tự hồi quy (tự chú ý nhân quả trên các từ đã sinh trước đó, chú ý chéo (cross-attention) vào các đặc trưng ảnh) sinh ra câu trả lời từng token một, chính xác là cơ chế đằng sau các trợ lý đa phương thức lớn có thể mô tả, đặt câu hỏi và suy luận về một ảnh bằng ngôn ngữ tự do thay vì chọn từ một danh sách cố định.

### Bài tập

1. Thêm một mẫu câu hỏi thứ năm hỏi về *vị trí* (ví dụ "có hình nào ở nửa trên của ảnh không", trả lời có/không dựa trên `cy < SIZE // 2`). Huấn luyện lại và kiểm tra xem độ chính xác trên loại câu hỏi mới này có khớp với các loại khác không — phép gộp cực đại toàn cục của bộ mã hóa ảnh (Bài 34) có làm cho các câu hỏi dựa trên vị trí khó trả lời hơn về mặt cấu trúc so với các câu hỏi về sự có mặt/số đếm không?
2. Thay phép hợp nhất nhân từng phần tử (`img_enc(img) * q_enc(q_ids)`) bằng phép nối theo sau là một lớp tuyến tính (`nn.Linear(2 * dim, dim)` trên `torch.cat([img_feat, q_feat], dim=-1)`). Độ chính xác có thay đổi thấy rõ không, và phương pháp hợp nhất nào hội tụ nhanh hơn trong khi huấn luyện?
3. Bộ mã hóa câu hỏi của bài này gộp trung bình các nhúng từ, nên `"is there a circle"` và `"there a circle is"` được mã hóa giống hệt nhau (bài tập ở Bài 52 đã nêu cùng điểm này về chú thích). Hãy dựng một cặp câu hỏi cho tập dữ liệu này mà ở đó sự mù thứ tự từ đó thực sự gây ra lỗi trả lời, nếu tồn tại một cặp như vậy — hoặc giải thích tại sao các mẫu câu hỏi của tập dữ liệu này đủ đơn giản để điều đó không bao giờ quan trọng.